# Lee-Carter Quickstart

This notebook demonstrates the full pyStMoMo workflow:
fit → forecast → simulate → bootstrap → plot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pystmomo as ps

plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
data = ps.load_ew_male()
print(f"Ages:  {data.ages[0]}–{data.ages[-1]}")
print(f"Years: {data.years[0]}–{data.years[-1]}")
print(f"Deaths shape: {data.deaths.shape}")

## 2. Fit Lee-Carter Model

In [ ]:
fit = ps.lc().fit(
    data.deaths, data.exposures,
    ages=data.ages, years=data.years,
)
print(fit)
print(f"AIC: {fit.aic():.1f}  BIC: {fit.bic():.1f}")

## 3. Visualise Parameters

In [ ]:
ps.plot_parameters(fit)
plt.tight_layout()
plt.show()

## 4. Residual Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ps.plot_residual_heatmap(fit, ax=axes[0])
ps.plot_residual_scatter(fit, ax=axes[1])
plt.tight_layout()
plt.show()

## 5. Forecast 20 Years

In [ ]:
fc = ps.forecast(fit, h=20)
print(f"Forecast rates shape: {fc.rates.shape}")
print(f"Forecast years: {fc.years_f[0]}–{fc.years_f[-1]}")

# Plot forecast for age 65
age_idx = np.where(data.ages == 65)[0][0]
plt.figure(figsize=(10, 4))
plt.semilogy(data.years, data.deaths[age_idx] / data.exposures[age_idx], 'k.', label='Observed')
plt.semilogy(fc.years_f, fc.rates[age_idx], 'b-', label='Forecast')
plt.semilogy(data.years, fit.fitted_rates[age_idx], 'r--', alpha=0.6, label='Fitted')
plt.xlabel('Year'); plt.ylabel('Mortality rate'); plt.title('Age 65')
plt.legend(); plt.show()

## 6. Simulate Trajectories & Fan Chart

In [ ]:
sim = ps.simulate(fit, nsim=1000, h=20, seed=42)
print(f"Simulation shape: {sim.rates.shape}")
ps.plot_fan(sim, age=65)
plt.show()

## 7. Bootstrap Parameter Uncertainty

In [ ]:
boot = ps.semiparametric_bootstrap(fit, nboot=100, seed=0)
print(boot)

lo, hi = boot.parameter_ci('kt', level=0.95)

plt.figure(figsize=(10, 4))
plt.plot(data.years, fit.kt[0], 'b-', label='κ_t (fitted)')
plt.fill_between(data.years, lo[0], hi[0], alpha=0.2, color='blue', label='95% CI')
plt.xlabel('Year'); plt.ylabel('κ_t'); plt.title('Period Index with Bootstrap CI')
plt.legend(); plt.show()